# Notebook 6: Real-Time Audio Decimation, High-Resolution FFT & Microphone Validation

This notebook verifies the **Phase 2.5 (`v1.4.5-rc1`)** hardware and software audio stack on the PYNQ-Z2.

### 🌟 Key Improvements in v1.4.5:
- 🎙 **Hardware PL Decimator ($M=10\times$):** Converts the raw 500 kSPS stream to an anti-aliased **50 kSPS audio stream**.
- ⏱ **Expanded $20.48\,\text{ms}$ Time Window:** Captures multiple full wave periods of bass ($50\,\text{Hz} - 250\,\text{Hz}$) and vocal frequencies.
- 📊 **High-Resolution Audio FFT ($\Delta f \approx 24.41\,\text{Hz}$):** Resolves individual musical notes and speech formants across $0 - 25\,\text{kHz}$.
- 🎯 **Sub-Bin Quadratic Peak Interpolation:** Delivers sub-Hertz frequency tracking accuracy ($\pm 0.2\,\text{Hz} - 0.5\,\text{Hz}$).
- 🎛 **Dedicated `AudioDashboard`:** Complete dark-mode instrument designed specifically for passive MAX4466 microphones with live clipping warning detectors.

## 1. System Setup & Overlay Initialization
Initialize `OscilloscopeOverlay()`. It automatically loads the `v1.4.5-rc1` bitstream containing the PL audio decimator.

In [ ]:
from pynq_oscilloscope import check_usb_permissions, OscilloscopeOverlay
from pynq_oscilloscope.fft_dma import StreamingFFT
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Ensure bus permissions
check_usb_permissions()

# Load Dual-Channel Decimated Audio Overlay
ol = OscilloscopeOverlay()
info = ol.set_profile("audio")

print("✅ Decimated Audio Overlay loaded successfully!")
print(f"  • Audio Sample Rate : {info['sample_rate_hz']/1e3:.1f} kSPS per channel")
print(f"  • Time Window       : {info['time_window_ms']:.2f} ms per 1024-sample frame")
print(f"  • FFT Bin Spacing   : Δf = {info['delta_f_hz']:.3f} Hz per bin (0 Hz to 25 kHz)")

## 2. MAX4466 Microphone Resting Bias & Saturation Check
Verify that the resting DC level of both microphones sits near the mid-rail ($1.65\,\text{V}$) and check that the signal is not clipping ($0.10\,\text{V} < V < 3.10\,\text{V}$).

In [ ]:
# Set Trigger to Center Baseline (1.65V) in Auto Mode
ol.trigger.configure(mode="Auto", edge="Rising", source="CH1", threshold_volts=1.65, timeout_ms=50.0)

# Capture a single stereo frame (1024 samples per channel)
v_a0, v_a1 = ol.capture_stereo()

mean_a0, vpp_a0 = float(np.mean(v_a0)), float(np.ptp(v_a0))
mean_a1, vpp_a1 = float(np.mean(v_a1)), float(np.ptp(v_a1))

print("🎙 Microphone Status Check:")
print(f"  • Mic 1 (A0): DC Offset = {mean_a0:.2f} V | Vpp = {vpp_a0:.2f} V | Min = {v_a0.min():.2f} V, Max = {v_a0.max():.2f} V")
print(f"  • Mic 2 (A1): DC Offset = {mean_a1:.2f} V | Vpp = {vpp_a1:.2f} V | Min = {v_a1.min():.2f} V, Max = {v_a1.max():.2f} V")

# Saturation warnings
if v_a0.min() < 0.10 or v_a0.max() > 3.10:
    print("⚠️ WARNING: Mic 1 (A0) is CLIPPING! Rotate the back trimmer counter-clockwise to reduce gain.")
else:
    print("✅ Mic 1 (A0) operating within linear dynamic range.")

if v_a1.min() < 0.10 or v_a1.max() > 3.10:
    print("⚠️ WARNING: Mic 2 (A1) is CLIPPING! Rotate the back trimmer counter-clockwise to reduce gain.")
else:
    print("✅ Mic 2 (A1) operating within linear dynamic range.")

## 3. Time-Domain Multi-Period Audio Waveform Capture
Speak, whistle, or play an acoustic test tone (e.g., $100\,\text{Hz}$ or $440\,\text{Hz}$) near the microphones. The plot below spans the full **$20.48\,\text{ms}$ time window**, clearly showing multiple complete wave cycles.

In [ ]:
import matplotlib.pyplot as plt

# Capture real-time acoustic waveforms
v_a0, v_a1 = ol.capture_stereo()
time_ms = np.linspace(0, 20.48, len(v_a0))

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 5.5), sharex=True, dpi=100)

ax1.plot(time_ms, v_a0, color="#00A389", linewidth=1.6, label="Mic 1: A0 (Vaux1)")
ax1.axhline(1.65, color="#FFA500", linestyle="--", alpha=0.7, label="DC Baseline (1.65V)")
ax1.set_title("Dual Microphone Acoustic Time-Domain Capture (20.48 ms Window @ 50 kSPS)", fontsize=11, fontweight="bold")
ax1.set_ylabel("Voltage (V)", fontsize=10)
ax1.set_ylim(0, 3.3)
ax1.grid(True, linestyle="--", alpha=0.5)
ax1.legend(loc="upper right")

ax2.plot(time_ms, v_a1, color="#D81B60", linewidth=1.6, label="Mic 2: A1 (Vaux9)")
ax2.axhline(1.65, color="#FFA500", linestyle="--", alpha=0.7, label="DC Baseline (1.65V)")
ax2.set_xlabel("Time (Milliseconds)", fontsize=10)
ax2.set_ylabel("Voltage (V)", fontsize=10)
ax2.set_ylim(0, 3.3)
ax2.grid(True, linestyle="--", alpha=0.5)
ax2.legend(loc="upper right")

plt.tight_layout()
plt.show()

## 4. High-Resolution Audio Spectrum & Sub-Bin Peak Tracking
Calculate the Hann-windowed frequency spectrum and extract the dominant fundamental frequency ($f_0$) using **Quadratic (Parabolic) Peak Interpolation**.

In [ ]:
# Capture audio frames
v_a0, v_a1 = ol.capture_stereo()
n = len(v_a0)

# 1. DC Baseline Removal & Hann Windowing
sig_a0 = (v_a0 - np.mean(v_a0)) * np.hanning(n)
sig_a1 = (v_a1 - np.mean(v_a1)) * np.hanning(n)

# 2. Fast Fourier Transform (0 to 25 kHz)
freqs = np.fft.rfftfreq(n, d=1.0 / 50000.0)
mag_a0 = 20.0 * np.log10(np.maximum(np.abs(np.fft.rfft(sig_a0)) / (n / 2.0), 1e-6))
mag_a1 = 20.0 * np.log10(np.maximum(np.abs(np.fft.rfft(sig_a1)) / (n / 2.0), 1e-6))

# 3. Sub-Bin Quadratic Peak Detection
peak_f0, peak_m0 = StreamingFFT.get_peak_frequency(freqs, mag_a0, min_freq_hz=20.0)
peak_f1, peak_m1 = StreamingFFT.get_peak_frequency(freqs, mag_a1, min_freq_hz=20.0)

print(f"🎯 Dominant Pitch on Mic 1 (A0): {peak_f0:.2f} Hz ({peak_m0:.1f} dBV)")
print(f"🎯 Dominant Pitch on Mic 2 (A1): {peak_f1:.2f} Hz ({peak_m1:.1f} dBV)")

# Plot Spectrum (Zoomed to 0 - 3000 Hz for voice and bass)
plt.figure(figsize=(10, 4.5), dpi=100)
plt.plot(freqs, mag_a0, color="#00A389", linewidth=1.6, label=f"Mic 1 Spectrum (Peak: {peak_f0:.1f} Hz)")
plt.plot(freqs, mag_a1, color="#D81B60", linewidth=1.4, alpha=0.8, label=f"Mic 2 Spectrum (Peak: {peak_f1:.1f} Hz)")
plt.scatter([peak_f0], [peak_m0], color="#FFA500", s=50, zorder=5, label="Fundamental Marker")

plt.title("High-Resolution Audio Frequency Spectrum (Δf = 24.41 Hz)", fontsize=11, fontweight="bold")
plt.xlabel("Frequency (Hz)", fontsize=10)
plt.ylabel("Magnitude (dBV)", fontsize=10)
plt.xlim(0, 3000)  # Audio Zoom
plt.ylim(-90, 0)
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend(loc="upper right")
plt.tight_layout()
plt.show()

## 5. Launch the Interactive Dedicated `AudioDashboard`
Launch the dedicated **`AudioDashboard`** for live multi-tab audio monitoring, real-time VU meter alerts, and acoustic spectrum analysis.

In [ ]:
# Launch Dedicated Audio & Microphone Dashboard
app = ol.audio_dashboard()

## 6. Hardware Shutdown & Cleanup
Stop the acquisition and release the contiguous DMA memory buffers.

In [ ]:
app.stop()
ol.close()
print("🔒 Audio acquisition stopped and hardware memory released.")